# Od DN k odrazivosti: Landsat 8 nad Hlubokou nad Vltavou

[![Otevřít v Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSkyLab-space/notebooks/blob/main/dn_to_reflectance_hluboka_cs.ipynb)

**OpenSkyLab · notebook k laboratoři** [Od DN k odrazivosti](https://openskylab.space/edu_viz_dn_to_reflectance.html) · [English version](dn_to_reflectance_hluboka_en.ipynb)

Laboratoř prochází řetězec od digitálního čísla přes zář k odrazivosti nad atmosférou (TOA) a na povrchu (BOA) a ukazuje, co opraví metoda tmavého objektu (DOS) a empirická přímka. Tady celý řetězec spočítáš na dvou skutečných snímcích Landsatu 8 nad Hlubokou nad Vltavou a výsledek porovnáš s odrazivostí Sentinelu-2 L2A z téhož týdne.

**Cíl:** převést DN na zář a odrazivost TOA, ukázat, proč DN ze dvou dnů nejsou srovnatelná, změřit vliv atmosféry na vodě a porovnat korekci DOS a empirickou přímku s odrazivostí na povrchu.

**Data:** Landsat 8 OLI Collection 1 Level-1 z veřejného archivu Google Cloud: `LC08_L1TP_191026_20190626_20190705_01_T1` (26. 6. 2019, výška Slunce 61°) a `LC08_L1TP_191026_20190218_20190222_01_T1` (18. 2. 2019, výška Slunce 27°), pásma B1–B7, BQA a MTL. Pro srovnání Sentinel-2 L2A `S2B_33UVQ_20190625_0_L2A` (25. 6. 2019) z Earth Search. Výřez 12 × 12 km kolem Hluboké nad Vltavou, asi 25 MB.

**Úkoly:** pět. Odpovědi piš do buněk *Tvoje odpověď* pod úkoly. Řádky označené `# ◀ parametr` můžeš měnit a buňku spustit znovu (Shift + Enter).

**Čas:** asi 45 min. Potřebuješ Google účet (Colab); nic se neinstaluje do počítače. Buňky spouštěj popořadě shora dolů.

## 1. Příprava a data

> **💻 Co dělá kód:** Doinstaluje knihovnu rasterio, která čte rastrová data (GeoTIFF) přímo z internetu. Ostatní knihovny už v Colabu jsou.

In [ ]:
%pip install -q rasterio

> **💻 Co dělá kód:** Pro letní a zimní scénu Landsatu 8 stáhne metadata (MTL) a ze všech sedmi optických pásem a masky kvality načte jen čtvercový výřez. Z masky určí pixely bez mraků a stínů a vypíše výšku Slunce, vzdálenost Země–Slunce a podíl jasných pixelů.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from rasterio.warp import Resampling, reproject

GCS = "https://storage.googleapis.com/gcp-public-data-landsat/LC08/01/191/026/"
SCENES = {
    "léto": "LC08_L1TP_191026_20190626_20190705_01_T1",
    "zima": "LC08_L1TP_191026_20190218_20190222_01_T1",
}
# Levý dolní roh výřezu v UTM 33N (m) a jeho strana.
X0, Y0, SIZE = 453090, 5427180, 12000
BANDS = {
    1: "pobřežní 443",
    2: "modrá 482",
    3: "zelená 561",
    4: "červená 655",
    5: "NIR 865",
    6: "SWIR1 1609",
    7: "SWIR2 2201",
}


def mtl(scene):
    txt = requests.get(f"{GCS}{scene}/{scene}_MTL.txt", timeout=60).text
    return {
        k.strip(): v.strip().strip('"')
        for k, v in (line.split(" = ") for line in txt.splitlines() if " = " in line)
    }


def read(scene, band):
    with rasterio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        GDAL_HTTP_MAX_RETRY=5,
        GDAL_HTTP_RETRY_DELAY=2,
    ):
        with rasterio.open(f"/vsicurl/{GCS}{scene}/{scene}_{band}.TIF") as r:
            w = (
                rasterio.windows.from_bounds(X0, Y0, X0 + SIZE, Y0 + SIZE, r.transform)
                .round_offsets()
                .round_lengths()
            )
            read.transform = r.window_transform(w)
            return r.read(1, window=w).astype(float)


L8 = {}
for season, scene in SCENES.items():
    meta = mtl(scene)
    dn = {b: read(scene, f"B{b}") for b in BANDS}
    L8_transform = read.transform
    qa = read(scene, "BQA").astype(int)
    # Bez mraků a jistých stínů.
    ok = ((qa >> 4) & 1 == 0) & ((qa >> 7) & 3 < 3) & (dn[1] > 0)
    L8[season] = {"meta": meta, "dn": dn, "ok": ok, "transform": L8_transform}
    print(
        f"{season}: {scene}, výška Slunce {float(meta['SUN_ELEVATION']):.1f}°, vzdálenost Země–Slunce {meta['EARTH_SUN_DISTANCE']} AU, jasné pixely {ok.mean():.1%}"
    )

## 2. Z DN na zář a odrazivost TOA

Zář: L = M_L · DN + A_L. Odrazivost TOA: ρ = (M_ρ · DN + A_ρ) / sin(výška Slunce). Koeficienty jsou v souboru MTL; v M_ρ je už započtena vzdálenost Země–Slunce i sluneční ozáření pásma.

> **💻 Co dělá kód:** Podle koeficientů z MTL převede digitální čísla každého pásma na zář a na odrazivost nad atmosférou (TOA) s opravou na výšku Slunce. Ukáže tabulku mediánů DN, záře a odrazivosti v létě a v zimě a jejich poměry.

In [ ]:
for s in L8.values():
    m, dn = s["meta"], s["dn"]
    sun = np.sin(np.radians(float(m["SUN_ELEVATION"])))
    s["rad"] = {
        b: float(m[f"RADIANCE_MULT_BAND_{b}"]) * dn[b]
        + float(m[f"RADIANCE_ADD_BAND_{b}"])
        for b in BANDS
    }
    s["toa"] = {
        b: (
            float(m[f"REFLECTANCE_MULT_BAND_{b}"]) * dn[b]
            + float(m[f"REFLECTANCE_ADD_BAND_{b}"])
        )
        / sun
        for b in BANDS
    }

rows = []
for b, name in BANDS.items():
    r = {"pásmo": f"B{b} {name}"}
    for season, s in L8.items():
        ok = s["ok"]
        r[f"DN {season}"] = np.median(s["dn"][b][ok])
        r[f"zář {season}"] = np.median(s["rad"][b][ok])
        r[f"ρ TOA {season}"] = np.median(s["toa"][b][ok])
    rows.append(r)
tab = pd.DataFrame(rows).set_index("pásmo")
tab["poměr DN léto/zima"] = tab["DN léto"] / tab["DN zima"]
tab["poměr ρ léto/zima"] = tab["ρ TOA léto"] / tab["ρ TOA zima"]
tab.round(3)

> **✏️ Úkol 1.** O kolik se mediány DN mezi létem a zimou liší a o kolik mediány odrazivosti TOA? Kolik z rozdílu DN vysvětlí jen výška Slunce (porovnej poměr sinů výšek)? Proč se ani odrazivost TOA mezi daty úplně neshoduje (stav porostu, případný sníh, atmosféra)?

*Tvoje odpověď:* 

## 3. Odrazivost na povrchu ze Sentinelu-2 jako reference

Sentinel-2 L2A z 25. 6. 2019 (o den dřív) má odrazivost BOA z modelu Sen2Cor. Převzorkujeme ho na mřížku Landsatu (průměr do pixelu 30 m) a přiřadíme si pásma s podobnou vlnovou délkou. Dokonalou shodu nečekej: jde o jiný senzor s trochu jinými pásmy, snímek o den dřív a z jiného úhlu pohledu.

> **💻 Co dělá kód:** Stáhne z katalogu Element 84 scénu Sentinelu-2 L2A s odrazivostí na povrchu (BOA) a jejích sedm pásem a mapu tříd SCL převzorkuje na mřížku Landsatu (průměrem do pixelu 30 m). Vybere pixely, kde jsou obě data platná, a vykreslí vedle sebe barevný snímek TOA z Landsatu a BOA ze Sentinelu-2 se stejným roztažením barev.

In [ ]:
item = requests.get(
    "https://earth-search.aws.element84.com/v1/collections/sentinel-2-l2a/items/S2B_33UVQ_20190625_0_L2A",
    timeout=60,
).json()
PAIR = {
    1: "coastal",
    2: "blue",
    3: "green",
    4: "red",
    5: "nir08",
    6: "swir16",
    7: "swir22",
}
shape = L8["léto"]["dn"][1].shape
grid = L8["léto"]["transform"]


def s2(key, res=Resampling.average):
    out = np.zeros(shape, np.float32)
    with rasterio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        GDAL_HTTP_MAX_RETRY=5,
        GDAL_HTTP_RETRY_DELAY=2,
    ):
        with rasterio.open("/vsicurl/" + item["assets"][key]["href"]) as r:
            reproject(
                rasterio.band(r, 1),
                out,
                dst_transform=grid,
                dst_crs="EPSG:32633",
                resampling=res,
            )
    return out


# Od základní linie zpracování 04.00 mají DN posun +1000, pokud ho katalog ještě
# neodečetl.
p = item["properties"]
off = (
    -1000
    if float(p.get("s2:processing_baseline", "0")) >= 4
    and not p.get("earthsearch:boa_offset_applied", False)
    else 0
)
boa = {b: (s2(k) + off) / 10000 for b, k in PAIR.items()}
scl = s2("scl", Resampling.mode)
summer = L8["léto"]
valid = summer["ok"] & np.isin(scl, [4, 5, 6]) & (boa[1] > 0)
print(f"pixelů pro srovnání: {valid.sum()}")
fig, ax = plt.subplots(1, 2, figsize=(11, 5))


def rgb(d):
    return np.clip(np.dstack([d[4], d[3], d[2]]) / 0.25, 0, 1)


ax[0].imshow(rgb(summer["toa"]))
ax[0].set_title("Landsat 8, odrazivost TOA, 26. 6. 2019")
ax[1].imshow(rgb(boa))
ax[1].set_title("Sentinel-2 L2A, odrazivost BOA, 25. 6. 2019")
for a in ax:
    a.axis("off")
plt.show()

> **✏️ Úkol 2.** Porovnej oba obrázky se stejným roztažením barev. Proč působí snímek s odrazivostí TOA světleji, modřeji a méně kontrastně? Které povrchy se mezi TOA a BOA mění nejvíc, tmavé, nebo světlé? Vysvětli to aditivním příspěvkem atmosféry.

*Tvoje odpověď:* 

## 4. Modrá voda nad atmosférou

Vltava a rybníky jsou v NIR skoro černé. Porovnáme jejich odrazivost TOA a BOA ve všech pásmech: rozdíl je hlavně světlo rozptýlené atmosférou (zář dráhy).

> **💻 Co dělá kód:** Vodu najde podle indexu NDWI ze Sentinelu-2 a třídy voda v SCL. Pro každé pásmo ukáže medián odrazivosti TOA a BOA nad vodou a jejich rozdíl, tedy příspěvek atmosféry.

In [ ]:
ndwi = (boa[3] - boa[5]) / (boa[3] + boa[5])
water = valid & (ndwi > 0.3) & (scl == 6)
w = pd.DataFrame(
    {
        "ρ TOA (Landsat)": [np.median(summer["toa"][b][water]) for b in BANDS],
        "ρ BOA (Sentinel-2)": [np.median(boa[b][water]) for b in BANDS],
    },
    index=[f"B{b} {n}" for b, n in BANDS.items()],
)
w["TOA − BOA"] = w.iloc[:, 0] - w.iloc[:, 1]
print(f"vodních pixelů: {water.sum()}")
w.round(4)

> **✏️ Úkol 3.** Jak se rozdíl TOA − BOA nad vodou mění s vlnovou délkou? Porovnej poměr mezi pásmy B1 a B4 s Rayleighovým zákonem (rozptyl ∝ λ⁻⁴). Proč je voda nad atmosférou modrá, i když je na povrchu tmavá ve všech pásmech?

*Tvoje odpověď:* 

## 5. Metoda tmavého objektu (DOS)

Předpokládáme, že nejtmavší pixely ve scéně mají na povrchu odrazivost skoro nulovou (běžně se bere 1 %, Chavez 1996). Co mají navíc, je zář dráhy, a tu odečteme od celé scény. DOS opraví jen aditivní část atmosféry.

> **💻 Co dělá kód:** Metoda tmavého objektu: v každém pásmu vezme odrazivost TOA na percentilu `pct` jako tmavý objekt a od celé scény odečte její rozdíl proti předpokládané odrazivosti `rho_dark`. Ukáže chybu (RMSE) původních dat a opravených dat proti Sentinelu-2 a průměrnou odchylku (bias).

In [ ]:
pct = 0.1  # ◀ parametr: percentil, který považujeme za tmavý objekt (%)
rho_dark = 0.01  # ◀ parametr: předpokládaná skutečná odrazivost tmavého objektu
dos = {
    b: summer["toa"][b]
    - (np.percentile(summer["toa"][b][summer["ok"]], pct) - rho_dark)
    for b in BANDS
}


def rmse(a, b, m=valid):
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))


def bias(a, b, m=valid):
    return float(np.mean(a[m] - b[m]))


res = pd.DataFrame(
    {
        "tmavý objekt ρ TOA": [
            np.percentile(summer["toa"][b][summer["ok"]], pct) for b in BANDS
        ],
        "RMSE TOA": [rmse(summer["toa"][b], boa[b]) for b in BANDS],
        "RMSE DOS": [rmse(dos[b], boa[b]) for b in BANDS],
        "bias DOS": [bias(dos[b], boa[b]) for b in BANDS],
    },
    index=[f"B{b}" for b in BANDS],
)
res.round(4)

> **✏️ Úkol 4.** Ve kterých pásmech DOS chybu proti BOA zmenší a ve kterých ji naopak zvětší? Proč je bias DOS ve všech pásmech záporný (co DOS dělá s násobnou složkou, propustností atmosféry)? Nastav `pct = 2` a pak `rho_dark = 0`: co se stane? Kdy DOS selže úplně (zimní scéna se sněhem bez tmavých objektů, opar jen nad částí scény)?

*Tvoje odpověď:* 

## 6. Empirická přímka

Empirická přímka opravuje odrazivost pomocí sklonu a posunu. V terénu mohou
jako reference sloužit fyzické kalibrační panely se známou odrazivostí.
V tomto cvičení použijeme **referenční pixely**: jejich referenční odrazivost
převezmeme ze snímku Sentinel-2 L2A.

Pro každé pásmo hledáme přímku mezi odrazivostí TOA Landsatu a referenční
odrazivostí BOA. Touto přímkou potom opravíme celé pásmo:

$$
\widehat{\rho}_{\mathrm{BOA}} = a\,\rho_{\mathrm{TOA}} + b.
$$

- $\rho_{\mathrm{TOA}}$ je vstupní odrazivost na horní hranici atmosféry.
- $\widehat{\rho}_{\mathrm{BOA}}$ je náš odhad odrazivosti na povrchu.
- $a$ je **sklon** (`slope`): mění měřítko hodnot.
- $b$ je **posun** (`offset`): ke každé hodnotě přičítá stejnou opravu.

Odrazivost, sklon i posun jsou bezrozměrné. Pro zjednodušený model atmosféry

$$
\rho_{\mathrm{TOA}} \approx \rho_{\mathrm{dráhy}} + T\,\rho_{\mathrm{BOA}}
\quad\Rightarrow\quad
a \approx \frac{1}{T},\qquad b \approx -\frac{\rho_{\mathrm{dráhy}}}{T},
$$

kde $T$ je propustnost atmosféry a $\rho_{\mathrm{dráhy}}$ příspěvek rozptylu
v atmosféře vyjádřený jako odrazivost. Tento fyzikální výklad je přibližný:
zde srovnáváme dva různé senzory a referenční BOA je sama výsledkem zpracování.

### 6.1 Vyber referenční pixely a oprav snímek

V každém pásmu vybereme podle Sentinelu-2 `n_reference` nejtmavších a stejný
počet nejsvětlejších platných pixelů. Při `n_reference = 300` tedy přímku
prokládáme **600 dvojicemi hodnot**, ne dvěma průměry skupin. Různá pásma
mohou vybrat různé pixely.

Zápis `[valid]` vybere hodnoty na společných platných místech obou snímků
ve stejném pořadí. `np.argsort` vrátí jejich indexy od nejtmavších po
nejsvětlejší. `np.polyfit(..., deg=1)` z vybraných dvojic vypočítá sklon
a posun přímky. Tu pak použijeme na celé pásmo Landsatu.

**Po změně `n_reference` spusť znovu všechny tři buňky této části (6.1–6.3).**

In [ ]:
# Počet tmavých i světlých referenčních pixelů; celkem 2× tolik.
n_reference = 300  # ◀ parametr: pixelů v každé skupině
empirical = {}
coefficients = {}

for band in BANDS:
    toa_image = summer["toa"][band]
    toa_valid = toa_image[valid]
    boa_valid = boa[band][valid]

    sorted_indices = np.argsort(boa_valid)
    dark_indices = sorted_indices[:n_reference]
    bright_indices = sorted_indices[-n_reference:]
    reference_indices = np.concatenate((dark_indices, bright_indices))

    toa_reference = toa_valid[reference_indices]
    boa_reference = boa_valid[reference_indices]
    slope, offset = np.polyfit(toa_reference, boa_reference, deg=1)

    empirical[band] = slope * toa_image + offset
    coefficients[band] = (slope, offset)

### 6.2 Porovnej chyby všech tří postupů

Tabulka ukáže sklon a posun v každém pásmu a RMSE původní odrazivosti TOA,
korekce DOS a empirické přímky proti referenci Sentinel-2 BOA:

$$
\mathrm{RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}
\left(\rho_i^{\mathrm{odhad}}-\rho_i^{\mathrm{reference}}\right)^2}.
$$

$N$ je počet všech platných pixelů a $i$ označuje jeden z nich. Menší RMSE
znamená lepší shodu s referencí. Všechny metody hodnotíme na stejných pixelech,
včetně těch použitých k proložení přímky; nejde tedy o nezávislé ověření přesnosti.

In [ ]:
rows = []
for band in BANDS:
    slope, offset = coefficients[band]
    rows.append(
        {
            "pásmo": f"B{band}",
            "sklon a": slope,
            "posun b": offset,
            "RMSE TOA": rmse(summer["toa"][band], boa[band]),
            "RMSE DOS": rmse(dos[band], boa[band]),
            "RMSE emp. přímka": rmse(empirical[band], boa[band]),
        }
    )

comparison = pd.DataFrame(rows).set_index("pásmo")
print(comparison.round(4))

### 6.3 Zobraz shodu se Sentinelem-2

Na vodorovné ose je referenční BOA, na svislé ose odrazivost daného postupu.
Černá přímka 1 : 1 označuje úplnou shodu. Barvy rozlišují modré pásmo B2,
červené B4 a blízké infračervené B5.

Pro přehlednost vykreslíme jen **4 000 náhodně vybraných platných pixelů**,
stejných pro všechny grafy i pásma. Pevné semínko `0` zajistí opakovatelný
výběr. Tento vzorek slouží pouze ke kreslení; kalibraci ani RMSE nemění.

In [ ]:
rng = np.random.default_rng(0)
n_valid = np.count_nonzero(valid)
plot_indices = rng.choice(n_valid, size=4000, replace=False)

methods = [("TOA", summer["toa"]), ("DOS", dos), ("empirická přímka", empirical)]
band_colors = {2: "tab:blue", 4: "tab:red", 5: "tab:purple"}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for axis, (name, estimates) in zip(axes, methods):
    for band, color in band_colors.items():
        reference_values = boa[band][valid]
        estimated_values = estimates[band][valid]
        axis.scatter(
            reference_values[plot_indices],
            estimated_values[plot_indices],
            s=2,
            alpha=0.4,
            c=color,
            label=f"B{band}",
        )
    axis.axline((0, 0), slope=1, color="k", lw=0.8)
    axis.set(
        xlim=(0, 0.5), ylim=(0, 0.5), xlabel="ρ BOA Sentinel-2", ylabel=name, title=name
    )

axes[0].legend()
plt.tight_layout()
plt.show()

> **✏️ Úkol 5.** Proč má empirická přímka v modrých pásmech záporný posun a sklon jiný než 1? Co fyzikálně vyjadřují? Která metoda má nejmenší RMSE v jednotlivých pásmech a proč DOS nemůže opravit sklon? Nastav `n_reference = 20` a znovu spusť buňky 6.1–6.3: jak se změní sklon, posun a RMSE? Proč v terénu stačí k určení přímky dva kalibrační panely s různou známou odrazivostí, ale musí ležet ve stejném snímku a za stejného osvětlení?

*Tvoje odpověď:* 

> **✏️ Rozšíření (nepovinné).** Porovnej výsledky se spolužákem, který spustil notebook pro jiné místo: vyšel podobný rozdíl TOA − BOA nad vodou a podobný sklon empirické přímky? Čím se mohou lišit (jiný den, jiný opar, jiná výška Slunce)?

*Tvoje odpověď:* 

---
### Zdroje dat a literatura

- Landsat 8 OLI Collection 1 Level-1, U.S. Geological Survey; veřejný archiv Google Cloud (`gcp-public-data-landsat`). Data USGS jsou bez omezení užití, s uvedením zdroje.
- USGS (2019). *Landsat 8 (L8) Data Users Handbook*, verze 5.0, kap. 5 (převod DN na zář a odrazivost TOA).
- Copernicus Sentinel-2 data 2019, ESA (L2A, Sen2Cor), registr AWS Open Data (`sentinel-cogs`), katalog Earth Search (Element 84). Volné užití s uvedením zdroje.
- Chavez, P. S. (1988). An improved dark-object subtraction technique for atmospheric scattering correction of multispectral data. *Remote Sensing of Environment* 24, 459–479. https://doi.org/10.1016/0034-4257(88)90019-3
- Chavez, P. S. (1996). Image-based atmospheric corrections – revisited and improved. *Photogrammetric Engineering and Remote Sensing* 62(9), 1025–1036.
- Smith, G. M., Milton, E. J. (1999). The use of the empirical line method to calibrate remotely sensed data to reflectance. *International Journal of Remote Sensing* 20, 2653–2662. https://doi.org/10.1080/014311699211994

### Jak citovat

OpenSkyLab (2026). Od DN k odrazivosti: Landsat 8 nad Hlubokou nad Vltavou [notebook, výukový materiál], verze 1.0. https://github.com/OpenSkyLab-space/notebooks/blob/main/dn_to_reflectance_hluboka_cs.ipynb. Licence CC BY 4.0.

Text a kód notebooku: OpenSkyLab, licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Data mají licence svých poskytovatelů (viz výše).

*Materiál připravil tým OpenSkyLab s pomocí generativní AI Claude (Anthropic), která pomáhala s návrhem kódu a textu. Za obsah odpovídají autoři.*